# Create flag parameter


In [0]:
dbutils.widgets.text('incremental_flag', '0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`

# Create dimension table

In [0]:
df_Dim_Model = spark.sql('''
SELECT DISTINCT(Model_ID) AS Model_ID, Model_Category FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`
''')


Dim_model sink for initial and incremental load

In [0]:
if spark.catalog.tableExists('sales_catalogue.gold.dim_model'):
   
    df_sink = spark.sql(''' 
                 SELECT Dim_model_key, Model_ID, Model_Category 
                 FROM sales_catalogue.gold.dim_model 
                 ''')
else:
   
    df_sink = spark.sql(''' 
                 SELECT 1 as Dim_model_key, Model_ID, Model_Category 
                 FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/` 
                 WHERE 1=0 
                 ''')

### filtering new and old records

In [0]:
df_filter = df_Dim_Model.join(df_sink, df_Dim_Model['Model_ID'] == df_sink['Model_ID'], 'left').select(df_sink['Dim_model_key'],df_Dim_Model['Model_ID'], df_Dim_Model['Model_Category'])

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

Df_filter_old

In [0]:
df_filter_old = df_filter.filter(col('Dim_model_key').isNotNull())


df_filter_new

In [0]:
df_filter_new = df_filter.filter(col('Dim_model_key').isNull())


# Create surrogate key

fetch the max surrogate key from the column from the existing table

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("select max(Dim_model_key) from sales_catalogue.gold.dim_model")
    max_value = max_value_df.collect()[0][0]

# Create surrogate key column and add the max surrogate key

In [0]:
df_filter_new = df_filter_new.withColumn('Dim_model_key', max_value+monotonically_increasing_id())

In [0]:
df_filter_new.display()


Dim_model_key,Model_ID,Model_Category


FINAL JOIN NEW + OLD

In [0]:
df_final = df_filter_old.union(df_filter_new)

In [0]:
df_final.display()

Dim_model_key,Model_ID,Model_Category
1,Mah-M167,Mah
2,Che-M47,Che
3,Toy-M205,Toy
4,BMW-M249,BMW
5,Mer-M122,Mer
6,Hon-M215,Hon
7,Nis-M82,Nis
8,Toy-M206,Toy
9,Mar-M139,Mar
10,Ren-M207,Ren


SCD- 1 Upsert

In [0]:
from delta.tables import *

In [0]:
if spark.catalog.tableExists('sales_catalog.gold.dim_model'):
    delta_table = deltaTable.forPath("abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales_dim_model")
    delta_table.alias("trg").merge(df_final.alias("src"), "trg.Dim_model_key = src.Dim_model_key")\
                             .whenMatchedUpdateAll()\
                             .whenNotMatchedInsertAll()\
                             .execute()

else: 
    df_final.write.format('delta')\
        .mode('overwrite')\
            .option("path", "abfss://gold-adf@rgstorage.dfs.core.windows.net/car_sales_dim_model")\
            .saveAsTable('sales_catalogue.gold.dim_model')

In [0]:
%sql
select * from sales_catalogue.gold.dim_model

Dim_model_key,Model_ID,Model_Category
1,Mah-M167,Mah
2,Che-M47,Che
3,Toy-M205,Toy
4,BMW-M249,BMW
5,Mer-M122,Mer
6,Hon-M215,Hon
7,Nis-M82,Nis
8,Toy-M206,Toy
9,Mar-M139,Mar
10,Ren-M207,Ren
